In [1]:
from dotenv import load_dotenv
import os
from langchain_groq import  ChatGroq

load_dotenv()

True

In [2]:
llm = ChatGroq(
    model_name = "openai/gpt-oss-20b",
    temperature = 0.0
)

In [44]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_prompt = """You are an official Airline Customer Support Assistant.

Your task is to answer the passenger's question accurately using ONLY the policy context provided below.

Strict Instructions:
1. Ground your answer strictly in the provided context. Do NOT use outside assumptions.
2. If the context does not contain the answer, politely reply:
   "I do not have specific policy information regarding this in my knowledge base. Please contact our airport desk or human support team for assistance."
3. Never invent fees, compensation amounts, or deadlines.
4. Keep the response polite, clear, and structured (use bullet points if explaining rules).

Context:
{context}
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

print("Prompt template created successfully!")

Prompt template created successfully!


In [4]:
test_prompt = prompt_template.invoke({
    "context": "Standard cabin baggage: 7 kg.\nAdditional personal article: Laptop bag up to 3 kg.",
    "question": "Can I bring a laptop bag in the cabin?"
})
print(test_prompt)

messages=[SystemMessage(content='You are an official Airline Customer Support Assistant.\n\nYour task is to answer the passenger\'s question accurately using ONLY the policy context provided below.\n\nStrict Instructions:\n1. Ground your answer strictly in the provided context. Do NOT use outside assumptions.\n2. If the context does not contain the answer, politely reply:\n   "I do not have specific policy information regarding this in my knowledge base. Please contact our airport desk or human support team for assistance."\n3. Never invent fees, compensation amounts, or deadlines.\n4. Keep the response polite, clear, and structured (use bullet points if explaining rules).\n\nContext:\nStandard cabin baggage: 7 kg.\nAdditional personal article: Laptop bag up to 3 kg.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='Can I bring a laptop bag in the cabin?', additional_kwargs={}, response_metadata={})]


In [5]:
test_prompt.messages

[SystemMessage(content='You are an official Airline Customer Support Assistant.\n\nYour task is to answer the passenger\'s question accurately using ONLY the policy context provided below.\n\nStrict Instructions:\n1. Ground your answer strictly in the provided context. Do NOT use outside assumptions.\n2. If the context does not contain the answer, politely reply:\n   "I do not have specific policy information regarding this in my knowledge base. Please contact our airport desk or human support team for assistance."\n3. Never invent fees, compensation amounts, or deadlines.\n4. Keep the response polite, clear, and structured (use bullet points if explaining rules).\n\nContext:\nStandard cabin baggage: 7 kg.\nAdditional personal article: Laptop bag up to 3 kg.\n', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Can I bring a laptop bag in the cabin?', additional_kwargs={}, response_metadata={})]

In [6]:
from langchain_community.embeddings import  HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

C:\Users\Omkar Raut\AppData\Local\Temp\ipykernel_30600\3991183308.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import  HuggingFaceEmbeddings
C:\Users\Omkar Raut\AppData\Local\Temp\ipykernel_30600\3991183308.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
c:\Users\Omkar Raut\OneDrive\Desktop\SIH-OIL\oil-sif\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and 

In [7]:
from langchain_community.vectorstores import  FAISS

vector_db = FAISS.load_local(
    "./vector-db",
    embedding_model,
    allow_dangerous_deserialization=True
)

In [19]:
for i in vector_db.similarity_search("Can I cary laptop as extra loggage"):
    print(i.metadata["source"])

knowledge-documents\01_baggage_policy.txt
knowledge-documents\01_baggage_policy.txt
knowledge-documents\01_baggage_policy.txt
knowledge-documents\03_passenger_rights_delays.txt


In [25]:
def ask_airline_assistant(question, k=3):

    documents = vector_db.similarity_search(question, k=k)

    context_text = "\n\n---\n\n".join(doc.page_content for doc in documents)

    format_prompt = prompt_template.invoke({
        "context": context_text,
        "question": question
    })

    response = llm.invoke(format_prompt)

    sources = list(set(doc.metadata.get("source", "Unknown") for doc in documents))

    return {
        "answer": response.content,
        "sources": sources
    }


In [26]:
# Test 1: Baggage Policy
res1 = ask_airline_assistant("Can I carry a laptop bag with my hand luggage, and what are the weight limits?")
print("=== RESPONSE ===")
print(res1["answer"])
print("\nSources:", res1["sources"])

=== RESPONSE ===
Yes – you may bring a laptop bag as an **additional personal article** alongside your main hand‑baggage piece.

**Weight limits**

- **Main hand‑baggage piece** – up to **7 kg**  
- **Additional personal article (e.g., laptop bag)** – up to **3 kg**

Both items must also meet the size restrictions for cabin baggage (55 cm × 35 cm × 25 cm, with a combined dimension limit of 115 cm). If either item exceeds the weight or size limits, it may be transferred to the checked‑baggage compartment, and applicable charges could apply.

Sources: ['knowledge-documents\\01_baggage_policy.txt']


In [27]:
# Test 2: Flight Delay Rights (DGCA)
res2 = ask_airline_assistant("My flight is delayed by 3 hours. Am I entitled to free meals?")
print("=== RESPONSE ===")
print(res2["answer"])
print("\nSources:", res2["sources"])

=== RESPONSE ===
According to the airline’s policy on delayed flights:

- **If your flight’s block time is up to 2.5 hours**  
  • A delay of **2 hours or more** entitles you to free meals and refreshments.

- **If your flight’s block time is between 2.5 and 5 hours**  
  • A delay of **3 hours or more** entitles you to free meals and refreshments.

- **If your flight’s block time is longer than 5 hours**  
  • A delay of **4 hours or more** is required for free meals and refreshments.

Since your flight is delayed by 3 hours, you would be entitled to free meals **provided the block time of your flight is 5 hours or less**. If the block time exceeds 5 hours, the policy requires a 4‑hour delay for meal entitlement.

Could you let me know the block time (total flight duration) for your flight? That will confirm whether the free‑meal entitlement applies.

Sources: ['knowledge-documents\\03_passenger_rights_delays.txt']


### Context History Based RAG

In [38]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1. Prompt that instructs the LLM to rewrite ambiguous follow-up questions
system_prompt = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

rephrase_chain = contextualize_q_prompt | llm

In [39]:
from langchain_core.messages import HumanMessage, AIMessage

# Simulate past conversation
mock_history = [
    HumanMessage(content="What is the cabin baggage limit on IndiGo?"),
    AIMessage(content="IndiGo allows 1 piece of cabin baggage up to 7 kg.")
]

# Follow-up question with ambiguous meaning
follow_up = "What about for infants?"

# Run the reformulator
rewritten_query = rephrase_chain.invoke({
    "chat_history": mock_history,
    "question": follow_up
})

print("ORIGINAL QUESTION :", follow_up)
print("REWRITTEN QUERY   :", rewritten_query.content)

ORIGINAL QUESTION : What about for infants?
REWRITTEN QUERY   : What is the cabin baggage limit for infants on IndiGo?


In [46]:
chat_history = []
def ask_airline_assistant_with_history(question, k=3):
    if not chat_history:
        rephrased_question = question
    else:
        rephrased_question = rephrase_chain.invoke({
            "chat_history": chat_history,
            "question": question
        }).content
    documents = vector_db.similarity_search(rephrased_question, k=k)

    context_text = "\n\n---\n\n".join(doc.page_content for doc in documents)

    format_prompt = prompt_template.invoke({
        "context": context_text,
        "chat_history": chat_history,
        "question": question
    })

    response = llm.invoke(format_prompt)

    sources = list(set(doc.metadata.get("source", "Unknown") for doc in documents))

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response.content))

    return {
        "answer": response.content,
        "sources": sources
    }

In [47]:
# Turn 1: Initial question
turn1 = ask_airline_assistant_with_history("What is the baggage allowance on IndiGo?")
print("AI (Turn 1):\n", turn1["answer"])
print("Sources:", turn1["sources"])

AI (Turn 1):
 **IndiGo Baggage Allowance – General Overview**

- **Two main categories**  
  - *Hand (cabin) baggage*  
  - *Checked‑in baggage*  

- **Allowance varies by**  
  - Fare type (e.g., Basic, Flex, etc.)  
  - Destination (domestic vs. international)  
  - Flight sector (specific route)  
  - Any special fare or codeshare arrangements  

- **How to find your exact allowance**  
  - Review the baggage details on your booking confirmation.  
  - Check your flight ticket or the “Baggage Allowance” section on IndiGo’s website.  

If you need the precise weight, size, or number of pieces for your specific flight, please refer to the links below or contact IndiGo customer support:

- [Baggage Policy](https://www.goindigo.in/information/baggage-policy.html)  
- [Baggage Allowance](https://www.goindigo.in/baggage/baggage-allowance.html)  
- [Cabin Baggage](https://www.goindigo.in/baggage/cabin-baggage.html)  
- [Excess Baggage](https://www.goindigo.in/baggage/excess-baggage.html)
S

In [48]:
# Turn 2: Follow-up relying on history
turn2 = ask_airline_assistant_with_history("What about for infants?")
print("AI (Turn 2):\n", turn2["answer"])
print("Sources:", turn2["sources"])

AI (Turn 2):
 **Infant Baggage Allowance on IndiGo**

- **Hand (cabin) baggage**  
  - One piece per infant.  
  - Must meet the standard 7 kg weight limit and dimension limits that apply to all hand baggage.

- **Checked‑in baggage**  
  - No separate checked‑in allowance is provided for infants.  
  - Infants are not entitled to an additional checked‑in bag beyond the standard allowance for the adult passenger.

- **Stroller / Baby Pram**  
  - One stroller or baby pram may be carried free of charge.  
  - It must comply with the airline’s conditions (e.g., size, weight, and handling requirements).

*Always confirm the specific rules for your booking by checking your confirmation or ticket, or by visiting IndiGo’s baggage policy pages.*
Sources: ['knowledge-documents\\01_baggage_policy.txt']


In [49]:
# Turn 3: Second follow-up
turn3 = ask_airline_assistant_with_history("Can I take a baby pram for free?")
print("AI (Turn 3):\n", turn3["answer"])
print("Sources:", turn3["sources"])

AI (Turn 3):
 Yes – IndiGo allows one stroller or baby pram to be carried free of charge.  
**Key points to keep in mind**

- **Free of charge** – no additional fee is applied for the pram.  
- **Compliance** – it must meet IndiGo’s size, weight, and handling requirements (e.g., it should be foldable and not exceed the maximum dimensions allowed for hand baggage).  
- **Check‑in** – bring it with you during check‑in; it is treated as part of your hand baggage allowance.  

If you’re unsure about the exact dimensions or weight limits, you can verify them on IndiGo’s website or by contacting customer support.
Sources: ['knowledge-documents\\01_baggage_policy.txt', 'knowledge-documents\\02_cancellation_refunds.txt', 'knowledge-documents\\04_checkin_boarding_seats.txt']


### n8n automation

In [51]:
import requests

web_hook_url = "https://intern.app.n8n.cloud/webhook-test/fa48a815-ee95-4229-a17e-5c38f6d28530"

data = {
    "customer_name": "Omkar",
    "query": "I want to cancel my flight",
    "intent": "cancel_flight"
}

response = requests.post(web_hook_url, json=data)

print(response.status_code)
print(response.text)

200
{"message":"Workflow was started"}


In [59]:
import requests

webhook_url = "https://intern.app.n8n.cloud/webhook-test/fa48a815-ee95-4229-a17e-5c38f6d28530"

data = {
    "customer_name": "Omkar",
    "customer_email": "omraut41105@gmail.com",
    "query": "What is the baggage allowance?",
    "intent": "check_baggage_allowance",
    "subject": "Your Airline Support Query",
    "answer": "Your baggage allowance depends on your airline and ticket type. Please check your booking details.",
    "resolved": False
}

response = requests.post(webhook_url, json=data)

print(response.status_code)
print(response.text)

200
{"message":"Workflow was started"}
